# Smart Desktop Keyboard — Grammar Engine (3-Model Comparison)
Exports & benchmarks **3 GEC models × 3 formats = 9 variants**.

| Model | HF ID | Prefix |
|---|---|---|
| prithivida | `prithivida/grammar_error_correcter_v1` | `gec: ` |
| coedit-small | `Unbabel/gec-t5_small` | `gec: ` |
| vennify-t5-base | `vennify/t5-base-grammar-correction` | `grammar: ` |

| Format | Description |
|---|---|
| **pytorch_fp32** | Original PyTorch weights, FP32 |
| **onnx_fp32** | ONNX export, no quantization |
| **onnx_int8** | ONNX + INT8 dynamic quantization |

### What this notebook produces
- Master comparison table (quality, latency, size for all 9 variants)
- Per-sentence regression & truncation analysis
- Latency CDF + bar charts
- Quality degradation heatmap (FP32 → INT8)
- Paired bootstrap significance tests (chrF++, 1000 resamples)
- Go/No-Go deployment scorecard
- Packaged INT8 models for download

**Runtime → CPU is fine. Run all cells top to bottom.**

## Step 1 — Install dependencies
Uninstalls conflicting Colab-preinstalled packages, then installs pinned versions.

**After this cell: Runtime → Restart session → continue from Step 2.**

In [ ]:
# ── Step 1 — Install all dependencies (Colab-safe) ────────────────────────────
# Colab's pre-installed packages (torch 2.10, transformers 5.x, huggingface_hub 0.30+,
# diffusers latest) are all incompatible with the older optimum/onnxruntime stack
# we need. Nuke everything and install a consistent, pinned set.

!pip uninstall -y torch torchvision torchaudio optimum transformers \
    onnxruntime onnx diffusers torchao huggingface_hub tokenizers 2>/dev/null

!pip install -q \
    "torch==2.2.0+cpu" \
    --extra-index-url https://download.pytorch.org/whl/cpu

!pip install -q \
    "numpy<2.0" \
    "huggingface_hub==0.23.4" \
    "tokenizers==0.19.1" \
    "transformers==4.41.0" \
    "diffusers==0.27.2" \
    "optimum[onnxruntime]==1.23.3" \
    "onnxruntime==1.18.0" \
    "onnx==1.16.0" \
    sentencepiece \
    sacrebleu \
    "protobuf==4.25.3" \
    psutil \
    tabulate \
    --no-build-isolation \
    --upgrade --upgrade-strategy eager

# ── Verify ────────────────────────────────────────────────────────────────────
import torch, transformers, sacrebleu, psutil
from importlib.metadata import version

print(f'torch           : {torch.__version__}')
print(f'transformers    : {transformers.__version__}')
print(f'optimum         : {version("optimum")}')
print(f'onnxruntime     : {version("onnxruntime")}')
print(f'onnx            : {version("onnx")}')
print(f'huggingface_hub : {version("huggingface_hub")}')
print(f'diffusers       : {version("diffusers")}')
print(f'numpy           : {version("numpy")}')
print(f'sacrebleu       : {sacrebleu.__version__}')
print()
print('All dependencies installed. ✓')
print('*** Restart runtime now, then continue from Step 2. ***')

Found existing installation: torch 2.10.0+cpu
Uninstalling torch-2.10.0+cpu:
  Successfully uninstalled torch-2.10.0+cpu
Found existing installation: torchvision 0.25.0+cpu
Uninstalling torchvision-0.25.0+cpu:
  Successfully uninstalled torchvision-0.25.0+cpu
Found existing installation: torchaudio 2.10.0+cpu
Uninstalling torchaudio-2.10.0+cpu:
  Successfully uninstalled torchaudio-2.10.0+cpu
Found existing installation: transformers 5.0.0
Uninstalling transformers-5.0.0:
  Successfully uninstalled transformers-5.0.0
Found existing installation: diffusers 0.37.1
Uninstalling diffusers-0.37.1:
  Successfully uninstalled diffusers-0.37.1
Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0
Found existing installation: huggingface_hub 1.8.0
Uninstalling huggingface_hub-1.8.0:
  Successfully uninstalled huggingface_hub-1.8.0
Found existing installation: tokenizers 0.22.2
Uninstalling tokenizers-0.22.2:
  Successfully uninstalled

## Step 2 — Shared constants & test data

In [ ]:
# ── Step 2 — Shared constants & test data ─────────────────────────────────────
import os

# ── 50-sentence validation set (complete sentences) ───────────────────────────
# Format: (incorrect_english, correct_english)
# Rules: model must fix grammar WITHOUT changing vocabulary or meaning
TEST_PAIRS = [
    ('I are going to office.',                        'I am going to office.'),
    ('She don\'t like the food.',                     'She doesn\'t like the food.'),
    ('He go to school every day.',                    'He goes to school every day.'),
    ('I has finished my work.',                       'I have finished my work.'),
    ('They was late yesterday.',                      'They were late yesterday.'),
    ('I am very boring today.',                       'I am very bored today.'),
    ('Please send me the informations.',              'Please send me the information.'),
    ('I have been working here since 3 years.',       'I have been working here for 3 years.'),
    ('She is more smarter than her brother.',         'She is smarter than her brother.'),
    ('I need a advise from you.',                     'I need advice from you.'),
    ('He told me to came early.',                     'He told me to come early.'),
    ('I am agree with your point.',                   'I agree with your point.'),
    ('We reached to the station on time.',            'We reached the station on time.'),
    ('She gave me an useful tip.',                    'She gave me a useful tip.'),
    ('I am waiting since morning.',                   'I have been waiting since morning.'),
    ('The meeting will held tomorrow.',               'The meeting will be held tomorrow.'),
    ('He is working hardly.',                         'He is working hard.'),
    ('I had went to the market.',                     'I had gone to the market.'),
    ('Please do the needful.',                        'Please do the needful.'),
    ('I am having a headache.',                       'I have a headache.'),
    ('She is knowing the answer.',                    'She knows the answer.'),
    ('I am understanding now.',                       'I understand now.'),
    ('Can you explain me this?',                      'Can you explain this to me?'),
    ('I will revert back to you.',                    'I will revert to you.'),
    ('The project is in advance stage.',              'The project is in an advanced stage.'),
    ('He don\'t have the document.',                  'He doesn\'t have the document.'),
    ('I want to discuss about this.',                 'I want to discuss this.'),
    ('She is my cousin sister.',                      'She is my cousin.'),
    ('We need to prepone the meeting.',               'We need to move the meeting earlier.'),
    ('I am out of station.',                          'I am out of station.'),
    ('He passed away last weak.',                     'He passed away last week.'),
    ('I have a good news for you.',                   'I have good news for you.'),
    ('Please do it at the earliest.',                 'Please do it at the earliest.'),
    ('I am not having any issues.',                   'I do not have any issues.'),
    ('The data are incorrect.',                       'The data is incorrect.'),
    ('She told that she will come.',                  'She said that she will come.'),
    ('I am based out of Mumbai.',                     'I am based out of Mumbai.'),
    ('He was very much tired.',                       'He was very tired.'),
    ('We are having a meeting tomorrow.',             'We are having a meeting tomorrow.'),
    ('I will ping you later.',                        'I will ping you later.'),
    ('The work is in full swing.',                    'The work is in full swing.'),
    ('She is elder than me.',                         'She is older than me.'),
    ('I need to do an important work.',               'I need to do an important task.'),
    ('Please revert at the earliest.',                'Please respond at the earliest.'),
    ('He is my childhood friend.',                    'He is my childhood friend.'),
    ('I will be there at sharp 3.',                   'I will be there at exactly 3.'),
    ('She was not in the meeting.',                   'She was not in the meeting.'),
    ('Please do the work as said.',                   'Please do the work as discussed.'),
    ('I am having second thoughts.',                  'I have second thoughts.'),
    ('He has went to Delhi.',                         'He has gone to Delhi.'),
]

# ── Partial / mid-sentence inputs (keyboard-specific edge cases) ──────────────
PARTIAL_INPUTS = [
    'I are',
    'She don',
    'Please send me the',
    'He told me to',
    'The meeting will',
    'I have been working here since',
    'Can you explain me',
]

assert len(TEST_PAIRS) == 50, f'Expected 50 test pairs, got {len(TEST_PAIRS)}'
print(f'Step 2 done. {len(TEST_PAIRS)} test pairs + {len(PARTIAL_INPUTS)} partial inputs loaded.')

Step 2 done. 50 test pairs + 7 partial inputs loaded.


## Step 3 — Model registry & directory layout

In [ ]:
# ── Prefix test for all new models ────────────────────────────────────────────
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

test_models = {
    "AventIQ-AI/T5-small-grammar-correction": ["grammar: ", "correct: ", "gec: ", ""],
    "visheratin/t5-efficient-mini-grammar-correction": ["grammar: ", "correct: ", "gec: ", ""],
    "visheratin/t5-efficient-tiny-grammar-correction": ["grammar: ", "correct: ", "gec: ", ""],
    "pszemraj/grammar-synthesis-small": ["grammar: ", "correct: ", "gec: ", "fix grammar: ", ""],
}

test_sentence = "She go to school yesterday"

for model_name, prefixes in test_models.items():
    print(f"\n{'='*70}")
    print(f"  {model_name}")
    print(f"{'='*70}")
    try:
        tok = AutoTokenizer.from_pretrained(model_name)
        model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
        model.eval()
        for prefix in prefixes:
            inp = tok([prefix + test_sentence], return_tensors="pt", max_length=128, truncation=True)
            out = model.generate(**inp, max_new_tokens=64)
            result = tok.decode(out[0], skip_special_tokens=True)
            print(f"  prefix='{prefix}' → {result}")
        del tok, model
    except Exception as e:
        print(f"  ❌ Failed to load: {e}")

The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

KeyboardInterrupt: 

In [ ]:
# ── Step 3 — Model registry & directory layout ───────────────────────────────
import os

MODEL_REGISTRY = {
    'prithivida': {
        'id':     'prithivida/grammar_error_correcter_v1',
        'prefix': 'gec: ',
    },
    'coedit-small': {
        'id':     'Unbabel/gec-t5_small',
        'prefix': 'gec: ',
    },
    'vennify-t5-base': {
        'id':     'vennify/t5-base-grammar-correction',
        'prefix': 'grammar: ',
    },
    'aventiq-t5-small': {
        'id':     'AventIQ-AI/T5-small-grammar-correction',
        'prefix': '',
    },
    'visheratin-mini': {
        'id':     'visheratin/t5-efficient-mini-grammar-correction',
        'prefix': '',
    },
    'visheratin-tiny': {
        'id':     'visheratin/t5-efficient-tiny-grammar-correction',
        'prefix': '',
    },
    'pszemraj-small': {
        'id':     'pszemraj/grammar-synthesis-small',
        'prefix': '',
    },
}

EXPORT_ROOT = '/content/gec_exports'

for mname in MODEL_REGISTRY:
    for fmt in ('onnx_fp32', 'onnx_int8'):
        os.makedirs(f'{EXPORT_ROOT}/{mname}/{fmt}', exist_ok=True)

print('Directory layout:')
for mname in MODEL_REGISTRY:
    print(f'  {EXPORT_ROOT}/{mname}/')
    for fmt in ('onnx_fp32', 'onnx_int8'):
        print(f'    {fmt}/')

print(f'\nRegistry ready. {len(MODEL_REGISTRY)} models × 3 formats = {len(MODEL_REGISTRY)*3} variants.')

Directory layout:
  /content/gec_exports/prithivida/
    onnx_fp32/
    onnx_int8/
  /content/gec_exports/coedit-small/
    onnx_fp32/
    onnx_int8/
  /content/gec_exports/vennify-t5-base/
    onnx_fp32/
    onnx_int8/
  /content/gec_exports/aventiq-t5-small/
    onnx_fp32/
    onnx_int8/
  /content/gec_exports/visheratin-mini/
    onnx_fp32/
    onnx_int8/
  /content/gec_exports/visheratin-tiny/
    onnx_fp32/
    onnx_int8/
  /content/gec_exports/pszemraj-small/
    onnx_fp32/
    onnx_int8/

Registry ready. 7 models × 3 formats = 21 variants.


## Step 4 — PyTorch FP32 baseline: all 3 models

In [ ]:
# ── Step 4 — Load & run all 3 PyTorch FP32 models ────────────────────────────
import torch, sacrebleu, time, os, psutil
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from tabulate import tabulate

all_src = [p[0] for p in TEST_PAIRS]
all_ref = [p[1] for p in TEST_PAIRS]
GEN_KWARGS = dict(num_beams=1, max_new_tokens=128, repetition_penalty=1.3)

def rss_mb():
    return psutil.Process(os.getpid()).memory_info().rss / 1e6

# Global results store: RESULTS[model_name][format] = {...}
RESULTS = {m: {} for m in MODEL_REGISTRY}

print('Loading PyTorch FP32 models ...\n')
for mname, cfg in MODEL_REGISTRY.items():
    print(f'  [{mname}]  {cfg["id"]}')
    r0    = rss_mb()
    tok   = AutoTokenizer.from_pretrained(cfg['id'])
    model = AutoModelForSeq2SeqLM.from_pretrained(cfg['id'])
    model.eval()
    rss_delta = rss_mb() - r0

    # Bulk inference
    prefixed = [cfg['prefix'] + s for s in all_src]
    inp = tok(prefixed, return_tensors='pt', padding=True, truncation=True, max_length=128)
    t0 = time.perf_counter()
    with torch.no_grad():
        out = model.generate(**inp, **GEN_KWARGS)
    bulk_s = time.perf_counter() - t0
    hyps = tok.batch_decode(out, skip_special_tokens=True)

    # Per-sentence latency
    lats = []
    for s in all_src:
        inp1 = tok([cfg['prefix'] + s], return_tensors='pt', truncation=True, max_length=128)
        t0 = time.perf_counter()
        with torch.no_grad():
            model.generate(**inp1, **GEN_KWARGS)
        lats.append((time.perf_counter() - t0) * 1000)

    lats_s = sorted(lats); n = len(lats_s)
    bleu = sacrebleu.corpus_bleu(hyps, [all_ref]).score
    chrf = sacrebleu.corpus_chrf(hyps, [all_ref]).score
    param_count = sum(p.numel() for p in model.parameters())

    RESULTS[mname]['pytorch_fp32'] = {
        'hyps': hyps, 'bleu': bleu, 'chrf': chrf,
        'p50': lats_s[n//2], 'p95': lats_s[int(n*.95)],
        'p99': lats_s[int(n*.99)], 'pmax': lats_s[-1],
        'throughput': len(all_src) / bulk_s,
        'rss_mb': rss_delta, 'params_m': param_count/1e6,
        'size_mb': param_count*4/1024**2, 'lats': lats,
    }
    print(f'    BLEU={bleu:.2f}  chrF++={chrf:.2f}  '
          f'P50={lats_s[n//2]:.0f}ms  DELTA_RSS={rss_delta:.0f}MB')
    del model, tok

rows = []
for mname in MODEL_REGISTRY:
    r = RESULTS[mname]['pytorch_fp32']
    rows.append([mname, f"{r['params_m']:.1f}", f"{r['size_mb']:.0f}",
                 f"{r['bleu']:.2f}", f"{r['chrf']:.2f}",
                 f"{r['p50']:.0f}", f"{r['p95']:.0f}",
                 f"{r['throughput']:.2f}", f"{r['rss_mb']:.0f}"])

print('\n── PyTorch FP32 Baseline ──')
print(tabulate(rows,
    headers=['Model','Params(M)','Size(MB)','BLEU','chrF++',
             'P50(ms)','P95(ms)','Tput(s/s)','DELTA_RSS(MB)'],
    tablefmt='github'))


The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

Loading PyTorch FP32 models ...

  [prithivida]  prithivida/grammar_error_correcter_v1


tokenizer_config.json: 0.00B [00:00, ?B/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


...

    BLEU=78.57  chrF++=87.87  P50=539ms  DELTA_RSS=505MB
  [aventiq-t5-small]  AventIQ-AI/T5-small-grammar-correction


Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
You set `add_prefix_space`. The tokenizer needs to be converted from the slow tokenizers


    BLEU=34.43  chrF++=45.88  P50=213ms  DELTA_RSS=3MB
  [visheratin-mini]  visheratin/t5-efficient-mini-grammar-correction
    BLEU=66.84  chrF++=81.77  P50=133ms  DELTA_RSS=51MB
  [visheratin-tiny]  visheratin/t5-efficient-tiny-grammar-correction
    BLEU=66.08  chrF++=82.69  P50=85ms  DELTA_RSS=33MB
  [pszemraj-small]  pszemraj/grammar-synthesis-small


/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:563: UserWarning: `num_beams` is set to 1. However, `early_stopping` is set to `True` -- this flag is only used in beam-based generation modes. You should set `num_beams>1` or unset `early_stopping`.
  warnings.warn(


    BLEU=57.11  chrF++=77.13  P50=363ms  DELTA_RSS=133MB

── PyTorch FP32 Baseline ──
| Model            |   Params(M) |   Size(MB) |   BLEU |   chrF++ |   P50(ms) |   P95(ms) |   Tput(s/s) |   DELTA_RSS(MB) |
|------------------|-------------|------------|--------|----------|-----------|-----------|-------------|-----------------|
| prithivida       |       222.9 |        850 |  76.88 |    86.4  |       602 |       801 |        6.2  |             938 |
| coedit-small     |        60.5 |        231 |  77.42 |    86.84 |       226 |       282 |       20.65 |              68 |
| vennify-t5-base  |       222.9 |        850 |  78.57 |    87.87 |       539 |       708 |        8.57 |             505 |
| aventiq-t5-small |        60.5 |        231 |  34.43 |    45.88 |       213 |       312 |       28.25 |               3 |
| visheratin-mini  |        31.2 |        119 |  66.84 |    81.77 |       133 |       194 |       64.91 |              51 |
| visheratin-tiny  |        15.6 |         59 

## Step 5 — Export all 3 models to ONNX FP32

In [ ]:
# ── Colab torch.xpu shim ─────────────────────────────────────────────────────
import types, torch
if not hasattr(torch, 'xpu') or not hasattr(torch.xpu, 'manual_seed'):
    class _FakeXPU(types.ModuleType):
        def is_available(self): return False
        def device_count(self): return 0
        def __getattr__(self, name):
            return lambda *a, **kw: None
    torch.xpu = _FakeXPU('torch.xpu')

# ── Force legacy ONNX exporter (bypass onnxscript dependency) ─────────────────
import shutil, os, torch
internal_path = os.path.join(os.path.dirname(torch.__file__), 'onnx', '_internal')
exporter_path = os.path.join(internal_path, 'exporter')

# Only remove the _compat.py file, NOT the whole exporter folder
compat_file = os.path.join(exporter_path, '_compat.py')
if os.path.exists(compat_file):
    os.remove(compat_file)
    print(f'Removed {compat_file}')

# Patch torch.onnx.export to use legacy path directly
import torch.onnx.utils
torch.onnx.export = torch.onnx.utils.export
print('Patched torch.onnx.export → legacy utils.export')

Removed /usr/local/lib/python3.12/dist-packages/torch/onnx/_internal/exporter/_compat.py
Patched torch.onnx.export → legacy utils.export


In [ ]:
# ── Colab torch.xpu shim ─────────────────────────────────────────────────────
import types, torch
if not hasattr(torch, 'xpu') or not hasattr(torch.xpu, 'manual_seed'):
    class _FakeXPU(types.ModuleType):
        def is_available(self): return False
        def device_count(self): return 0
        def __getattr__(self, name):
            return lambda *a, **kw: None
    torch.xpu = _FakeXPU('torch.xpu')

# ── Force legacy ONNX exporter (bypass onnxscript dependency) ─────────────────
import shutil, os
internal_path = os.path.join(os.path.dirname(torch.__file__), 'onnx', '_internal', 'exporter')
if os.path.exists(internal_path):
    shutil.rmtree(internal_path)
    print(f'Removed {internal_path} (forces legacy ONNX exporter)')

# ── Step 5 — ONNX FP32 export: all 3 models ─────────────────────────────────
import os, onnx, sacrebleu, time, psutil
from optimum.onnxruntime import ORTModelForSeq2SeqLM
from transformers import AutoTokenizer
from onnxruntime import SessionOptions
from tabulate import tabulate

GRAPHS = ['encoder_model.onnx', 'decoder_model.onnx', 'decoder_with_past_model.onnx']
GEN_KWARGS = dict(num_beams=1, max_new_tokens=128, repetition_penalty=1.3)
all_src = [p[0] for p in TEST_PAIRS]
all_ref = [p[1] for p in TEST_PAIRS]

def make_sess(n=2):
    s = SessionOptions()
    s.intra_op_num_threads = n
    s.inter_op_num_threads = 1
    return s

size_rows = []

for mname, cfg in MODEL_REGISTRY.items():
    out_dir = f'{EXPORT_ROOT}/{mname}/onnx_fp32'
    print(f'\n── Exporting [{mname}] to ONNX FP32 ──')

    ort_m = ORTModelForSeq2SeqLM.from_pretrained(cfg['id'], export=True)
    ort_m.save_pretrained(out_dir)
    tok = AutoTokenizer.from_pretrained(cfg['id'])
    tok.save_pretrained(out_dir)

    for g in GRAPHS:
        path = f'{out_dir}/{g}'
        if not os.path.exists(path): continue
        m = onnx.load(path)
        onnx.checker.check_model(m)
        mb = os.path.getsize(path) / 1e6
        size_rows.append([mname, 'onnx_fp32', g.replace('_model.onnx',''), f'{mb:.1f}', len(m.graph.node)])
        print(f'  {g:<45} {mb:6.1f} MB  nodes={len(m.graph.node)}  ok')

    r0 = rss_mb()
    tok2   = AutoTokenizer.from_pretrained(out_dir)
    ort_m2 = ORTModelForSeq2SeqLM.from_pretrained(out_dir, session_options=make_sess())
    rss_delta = rss_mb() - r0

    prefixed = [cfg['prefix'] + s for s in all_src]
    inp = tok2(prefixed, return_tensors='pt', padding=True, truncation=True, max_length=128)
    t0 = time.perf_counter()
    out = ort_m2.generate(**inp, **GEN_KWARGS)
    bulk_s = time.perf_counter() - t0
    hyps = tok2.batch_decode(out, skip_special_tokens=True)

    # Warmup then per-sentence latency
    for s in all_src[:2]:
        inp1 = tok2([cfg['prefix'] + s], return_tensors='pt', truncation=True, max_length=128)
        ort_m2.generate(**inp1, **GEN_KWARGS)
    lats = []
    for s in all_src:
        inp1 = tok2([cfg['prefix'] + s], return_tensors='pt', truncation=True, max_length=128)
        t0 = time.perf_counter()
        ort_m2.generate(**inp1, **GEN_KWARGS)
        lats.append((time.perf_counter() - t0) * 1000)

    lats_s = sorted(lats); n = len(lats_s)
    bleu = sacrebleu.corpus_bleu(hyps, [all_ref]).score
    chrf = sacrebleu.corpus_chrf(hyps, [all_ref]).score
    total_mb = sum(os.path.getsize(f'{out_dir}/{f}') for f in os.listdir(out_dir)) / 1e6

    RESULTS[mname]['onnx_fp32'] = {
        'hyps': hyps, 'bleu': bleu, 'chrf': chrf,
        'p50': lats_s[n//2], 'p95': lats_s[int(n*.95)],
        'p99': lats_s[int(n*.99)], 'pmax': lats_s[-1],
        'throughput': len(all_src)/bulk_s,
        'rss_mb': rss_delta, 'disk_mb': total_mb, 'lats': lats,
    }
    print(f'  BLEU={bleu:.2f}  chrF++={chrf:.2f}  '
          f'P50={lats_s[n//2]:.0f}ms  disk={total_mb:.0f}MB  DELTA_RSS={rss_delta:.0f}MB')
    del ort_m, ort_m2, tok, tok2

print('\n── Per-graph sizes (ONNX FP32) ──')
print(tabulate(size_rows, headers=['Model','Format','Graph','MB','Nodes'], tablefmt='github'))
print('\nAll ONNX FP32 exports complete.')

Removed /usr/local/lib/python3.12/dist-packages/torch/onnx/_internal/exporter (forces legacy ONNX exporter)

── Exporting [prithivida] to ONNX FP32 ──


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/modeling_utils.py:1018: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  if causal_mask.shape[1] < attention_mask.shape[1]:
/usr/local/lib/python3.12/dist-packag

  encoder_model.onnx                             438.7 MB  nodes=912  ok
  decoder_model.onnx                             650.8 MB  nodes=1658  ok
  decoder_with_past_model.onnx                   594.2 MB  nodes=1509  ok
  BLEU=76.88  chrF++=86.40  P50=517ms  disk=1687MB  DELTA_RSS=3167MB

── Exporting [coedit-small] to ONNX FP32 ──


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/modeling_utils.py:1018: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  if causal_mask.shape[1] < attention_mask.shape[1]:
/usr/local/lib/python3.12/dist-packag

  encoder_model.onnx                             141.4 MB  nodes=492  ok
  decoder_model.onnx                             232.5 MB  nodes=896  ok
  decoder_with_past_model.onnx                   219.9 MB  nodes=843  ok
  BLEU=77.42  chrF++=86.84  P50=202ms  disk=597MB  DELTA_RSS=291MB

── Exporting [vennify-t5-base] to ONNX FP32 ──


...

  encoder_model.onnx                              45.5 MB  nodes=352  ok
  decoder_model.onnx                              82.7 MB  nodes=642  ok
  decoder_with_past_model.onnx                    80.6 MB  nodes=621  ok
  BLEU=66.08  chrF++=82.69  P50=97ms  disk=211MB  DELTA_RSS=115MB

── Exporting [pszemraj-small] to ONNX FP32 ──


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Some non-default generation parameters are set in the model config. These should go into a GenerationConfig file (https://huggingface.co/docs/transformers/generation_strategies#save-a-custom-decoding-strategy-with-your-model) instead. This warning will be raised to an exception in v4.41.
Non-default generation parameters: {'max_length': 512, 'min_length': 8, 'early_stopping': True, 'num_beams': 2, 'no_repeat_ngram_size': 4}
/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr

  encoder_model.onnx                             141.5 MB  nodes=744  ok
  decoder_model.onnx                             232.6 MB  nodes=1260  ok
  decoder_with_past_model.onnx                   220.0 MB  nodes=1175  ok


/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:563: UserWarning: `num_beams` is set to 1. However, `early_stopping` is set to `True` -- this flag is only used in beam-based generation modes. You should set `num_beams>1` or unset `early_stopping`.
  warnings.warn(


  BLEU=57.11  chrF++=77.13  P50=236ms  disk=597MB  DELTA_RSS=232MB

── Per-graph sizes (ONNX FP32) ──
| Model            | Format    | Graph             |    MB |   Nodes |
|------------------|-----------|-------------------|-------|---------|
| prithivida       | onnx_fp32 | encoder           | 438.7 |     912 |
| prithivida       | onnx_fp32 | decoder           | 650.8 |    1658 |
| prithivida       | onnx_fp32 | decoder_with_past | 594.2 |    1509 |
| coedit-small     | onnx_fp32 | encoder           | 141.4 |     492 |
| coedit-small     | onnx_fp32 | decoder           | 232.5 |     896 |
| coedit-small     | onnx_fp32 | decoder_with_past | 219.9 |     843 |
| vennify-t5-base  | onnx_fp32 | encoder           | 438.7 |     912 |
| vennify-t5-base  | onnx_fp32 | decoder           | 650.8 |    1658 |
| vennify-t5-base  | onnx_fp32 | decoder_with_past | 594.2 |    1509 |
| aventiq-t5-small | onnx_fp32 | encoder           | 141.4 |     492 |
| aventiq-t5-small | onnx_fp32 | decoder      

## Step 6 — INT8 dynamic quantization: all 3 models

In [ ]:
# ── Step 6 — INT8 dynamic quantization: all 3 models ─────────────────────────
import glob, shutil, os, onnx, sacrebleu, time, psutil
from onnxruntime.quantization import quantize_dynamic, QuantType
from onnxruntime import SessionOptions
from optimum.onnxruntime import ORTModelForSeq2SeqLM
from transformers import AutoTokenizer
from tabulate import tabulate
from collections import Counter

GEN_KWARGS = dict(num_beams=1, max_new_tokens=128, repetition_penalty=1.3)
all_src = [p[0] for p in TEST_PAIRS]
all_ref = [p[1] for p in TEST_PAIRS]

def make_sess(n=2):
    s = SessionOptions()
    s.intra_op_num_threads = n
    s.inter_op_num_threads = 1
    return s

quant_rows = []

for mname, cfg in MODEL_REGISTRY.items():
    src_dir = f'{EXPORT_ROOT}/{mname}/onnx_fp32'
    dst_dir = f'{EXPORT_ROOT}/{mname}/onnx_int8'
    print(f'\n── Quantizing [{mname}] to INT8 ──')

    for src_path in sorted(glob.glob(f'{src_dir}/*.onnx')):
        fname    = os.path.basename(src_path)
        dst_path = f'{dst_dir}/{fname}'
        print(f'  {fname} ...', end=' ', flush=True)
        quantize_dynamic(src_path, dst_path, weight_type=QuantType.QInt8)
        m   = onnx.load(dst_path)
        ctr = Counter(node.op_type for node in m.graph.node)
        n_mmi   = ctr.get('MatMulInteger', 0)
        n_total = sum(ctr.values())
        if n_mmi == 0:
            print('WARNING: 0 MatMulInteger nodes!')
        else:
            print(f'MatMulInteger={n_mmi}/{n_total}  ok')
        mb = os.path.getsize(dst_path) / 1e6
        quant_rows.append([mname, fname.replace('_model.onnx',''), f'{mb:.1f}', n_mmi, n_total])

    for f in os.listdir(src_dir):
        if not f.endswith('.onnx'):
            shutil.copy2(f'{src_dir}/{f}', f'{dst_dir}/{f}')

    total_mb = sum(os.path.getsize(f'{dst_dir}/{f}') for f in os.listdir(dst_dir)) / 1e6
    print(f'  Total on disk: {total_mb:.0f} MB')

    r0 = rss_mb()
    tok   = AutoTokenizer.from_pretrained(dst_dir)
    ort_m = ORTModelForSeq2SeqLM.from_pretrained(dst_dir, session_options=make_sess())
    rss_delta = rss_mb() - r0

    prefixed = [cfg['prefix'] + s for s in all_src]
    inp = tok(prefixed, return_tensors='pt', padding=True, truncation=True, max_length=128)
    t0 = time.perf_counter()
    out = ort_m.generate(**inp, **GEN_KWARGS)
    bulk_s = time.perf_counter() - t0
    hyps = tok.batch_decode(out, skip_special_tokens=True)

    for s in all_src[:2]:
        inp1 = tok([cfg['prefix'] + s], return_tensors='pt', truncation=True, max_length=128)
        ort_m.generate(**inp1, **GEN_KWARGS)
    lats = []
    for s in all_src:
        inp1 = tok([cfg['prefix'] + s], return_tensors='pt', truncation=True, max_length=128)
        t0 = time.perf_counter()
        ort_m.generate(**inp1, **GEN_KWARGS)
        lats.append((time.perf_counter() - t0) * 1000)

    lats_s = sorted(lats); n = len(lats_s)
    bleu = sacrebleu.corpus_bleu(hyps, [all_ref]).score
    chrf = sacrebleu.corpus_chrf(hyps, [all_ref]).score

    RESULTS[mname]['onnx_int8'] = {
        'hyps': hyps, 'bleu': bleu, 'chrf': chrf,
        'p50': lats_s[n//2], 'p95': lats_s[int(n*.95)],
        'p99': lats_s[int(n*.99)], 'pmax': lats_s[-1],
        'throughput': len(all_src)/bulk_s,
        'rss_mb': rss_delta, 'disk_mb': total_mb, 'lats': lats,
    }
    print(f'  BLEU={bleu:.2f}  chrF++={chrf:.2f}  '
          f'P50={lats_s[n//2]:.0f}ms  disk={total_mb:.0f}MB  DELTA_RSS={rss_delta:.0f}MB')
    del ort_m, tok

print('\n── Quantization summary (INT8, per graph) ──')
print(tabulate(quant_rows,
    headers=['Model','Graph','MB','MatMulInteger','TotalNodes'], tablefmt='github'))
print('\nAll INT8 quantizations complete.')



── Quantizing [prithivida] to INT8 ──
  decoder_model.onnx ... 

MatMulInteger=121/2097  ok
  decoder_with_past_model.onnx ... 

MatMulInteger=97/1875  ok
  encoder_model.onnx ... 

MatMulInteger=72/1178  ok
  Total on disk: 426 MB
  BLEU=74.44  chrF++=84.79  P50=222ms  disk=426MB  DELTA_RSS=484MB

── Quantizing [coedit-small] to INT8 ──
  decoder_model.onnx ... 

MatMulInteger=61/1119  ok
  decoder_with_past_model.onnx ... 

...

MatMulInteger=33/747  ok
  encoder_model.onnx ... 

MatMulInteger=24/442  ok
  Total on disk: 55 MB
  BLEU=64.51  chrF++=82.28  P50=60ms  disk=55MB  DELTA_RSS=33MB

── Quantizing [pszemraj-small] to INT8 ──
  decoder_model.onnx ... 

MatMulInteger=89/1579  ok
  decoder_with_past_model.onnx ... 

MatMulInteger=73/1445  ok
  encoder_model.onnx ... 

MatMulInteger=56/946  ok
  Total on disk: 153 MB
  BLEU=3.15  chrF++=18.72  P50=138ms  disk=153MB  DELTA_RSS=284MB

── Quantization summary (INT8, per graph) ──
| Model            | Graph             |    MB |   MatMulInteger |   TotalNodes |
|------------------|-------------------|-------|-----------------|--------------|
| prithivida       | decoder           | 163.3 |             121 |         2097 |
| prithivida       | decoder_with_past | 149.1 |              97 |         1875 |
| prithivida       | encoder           | 110   |              72 |         1178 |
| coedit-small     | decoder           |  58.4 |              61 |         1119 |
| coedit-small     | decoder_with_past |  55.2 |              49 |         1029 |
| coedit-small     | encoder           |  35.5 |              36 |          626 |
| vennify-t5-base  | decoder           | 163.3 |             121 |         2097 |
| vennify-t5-base  | decoder_with_past | 149.1 |              97 |         1875 |
| vennify-t5-base  

## Step 7 — Master comparison table (all 9 variants)

In [ ]:
# ── Step 7 — Master comparison table ─────────────────────────────────────────
from tabulate import tabulate

FORMATS = ['pytorch_fp32', 'onnx_fp32', 'onnx_int8']

# ── Quality ───────────────────────────────────────────────────────────────────
print('=' * 72)
print('QUALITY — BLEU / chrF++ (all 9 variants, delta vs pytorch_fp32 baseline)')
print('=' * 72)
rows = []
for mname in MODEL_REGISTRY:
    for fmt in FORMATS:
        r   = RESULTS[mname].get(fmt)
        ref = RESULTS[mname]['pytorch_fp32']
        if r is None: continue
        rows.append([mname, fmt,
                     f"{r['bleu']:.2f}", f"{r['chrf']:.2f}",
                     f"{r['bleu']-ref['bleu']:+.2f}",
                     f"{r['chrf']-ref['chrf']:+.2f}"])
print(tabulate(rows,
    headers=['Model','Format','BLEU','chrF++','DELTA_BLEU','DELTA_chrF++'],
    tablefmt='github'))

# ── Latency ───────────────────────────────────────────────────────────────────
print('\n' + '=' * 72)
print('LATENCY — per-sentence ms (2-thread CPU, after 2 warmup calls)')
print('=' * 72)
rows = []
for mname in MODEL_REGISTRY:
    for fmt in FORMATS:
        r = RESULTS[mname].get(fmt)
        if r is None: continue
        rows.append([mname, fmt,
                     f"{r['p50']:.0f}", f"{r['p95']:.0f}",
                     f"{r['p99']:.0f}", f"{r['pmax']:.0f}",
                     f"{r['throughput']:.2f}"])
print(tabulate(rows,
    headers=['Model','Format','P50(ms)','P95(ms)','P99(ms)','Max(ms)','Tput(s/s)'],
    tablefmt='github'))

# ── Size & Memory ─────────────────────────────────────────────────────────────
print('\n' + '=' * 72)
print('SIZE & MEMORY')
print('=' * 72)
rows = []
for mname in MODEL_REGISTRY:
    pt = RESULTS[mname]['pytorch_fp32']
    for fmt in FORMATS:
        r    = RESULTS[mname].get(fmt)
        if r is None: continue
        disk = r.get('disk_mb', pt['size_mb'])
        rows.append([mname, fmt, f"{disk:.0f}", f"{r['rss_mb']:.0f}"])
print(tabulate(rows,
    headers=['Model','Format','Disk(MB)','DELTA_RSS(MB)'],
    tablefmt='github'))


QUALITY — BLEU / chrF++ (all 9 variants, delta vs pytorch_fp32 baseline)
| Model            | Format       |   BLEU |   chrF++ |   DELTA_BLEU |   DELTA_chrF++ |
|------------------|--------------|--------|----------|--------------|----------------|
| prithivida       | pytorch_fp32 |  76.88 |    86.4  |         0    |           0    |
| prithivida       | onnx_fp32    |  76.88 |    86.4  |         0    |           0    |
| prithivida       | onnx_int8    |  74.44 |    84.79 |        -2.44 |          -1.61 |
| coedit-small     | pytorch_fp32 |  77.42 |    86.84 |         0    |           0    |
| coedit-small     | onnx_fp32    |  77.42 |    86.84 |         0    |           0    |
| coedit-small     | onnx_int8    |  77.6  |    86.69 |         0.18 |          -0.15 |
| vennify-t5-base  | pytorch_fp32 |  78.57 |    87.87 |         0    |           0    |
| vennify-t5-base  | onnx_fp32    |  78.57 |    87.87 |         0    |           0    |
| vennify-t5-base  | onnx_int8    |  77.24 |   

## Step 8 — Per-sentence exact-match, regression & truncation analysis

In [ ]:
# ── Step 8 — Per-sentence analysis ───────────────────────────────────────────
from tabulate import tabulate

all_src = [p[0] for p in TEST_PAIRS]
all_ref = [p[1] for p in TEST_PAIRS]
FORMATS = ['pytorch_fp32', 'onnx_fp32', 'onnx_int8']

# ── Exact-match rate ──────────────────────────────────────────────────────────
print('── Exact-match rate (hyp == ref, case-insensitive) ──')
em_rows = []
for mname in MODEL_REGISTRY:
    for fmt in FORMATS:
        r = RESULTS[mname].get(fmt)
        if r is None: continue
        em = sum(1 for h, ref in zip(r['hyps'], all_ref)
                 if h.strip().lower() == ref.strip().lower())
        em_rows.append([mname, fmt, f'{em}/{len(all_ref)}', f'{100*em/len(all_ref):.1f}%'])
print(tabulate(em_rows, headers=['Model','Format','Exact','Rate'], tablefmt='github'))

# ── INT8 regressions vs PyTorch FP32 ─────────────────────────────────────────
print('\n── INT8 regressions vs pytorch_fp32 (sentences that changed output) ──')
for mname in MODEL_REGISTRY:
    pt_hyps = RESULTS[mname]['pytorch_fp32']['hyps']
    i8      = RESULTS[mname].get('onnx_int8')
    if i8 is None: continue
    i8_hyps = i8['hyps']
    regressions = [
        (src, pt_h, i8_h, ref)
        for src, pt_h, i8_h, ref in zip(all_src, pt_hyps, i8_hyps, all_ref)
        if pt_h.strip().lower() != i8_h.strip().lower()
    ]
    print(f'\n[{mname}]  {len(regressions)} sentence(s) differ (INT8 vs PT-FP32)')
    for src, pt_h, i8_h, ref in regressions[:5]:
        pt_ok = 'OK' if pt_h.strip().lower() == ref.strip().lower() else '~~'
        i8_ok = 'OK' if i8_h.strip().lower() == ref.strip().lower() else '~~'
        print(f'  SRC      : {src}')
        print(f'  PT  [{pt_ok}] : {pt_h}')
        print(f'  INT8[{i8_ok}] : {i8_h}')
        print(f'  REF      : {ref}')
        print()

# ── Truncation check ──────────────────────────────────────────────────────────
print('── Truncation check (output < 30% of input char length) ──')
trunc_rows = []
for mname in MODEL_REGISTRY:
    for fmt in FORMATS:
        r = RESULTS[mname].get(fmt)
        if r is None: continue
        bad = [(s, h) for s, h in zip(all_src, r['hyps']) if len(h) < len(s) * 0.3]
        trunc_rows.append([mname, fmt, len(bad), 'WARNING' if bad else 'OK'])
print(tabulate(trunc_rows,
    headers=['Model','Format','Truncated','Status'], tablefmt='github'))

# ── Partial input robustness ──────────────────────────────────────────────────
print('\n── Partial-input robustness (keyboard mid-type fragments) ──')
from optimum.onnxruntime import ORTModelForSeq2SeqLM
from transformers import AutoTokenizer
from onnxruntime import SessionOptions

for mname, cfg in MODEL_REGISTRY.items():
    dst_dir = f'{EXPORT_ROOT}/{mname}/onnx_int8'
    sess = SessionOptions()
    sess.intra_op_num_threads = 2
    sess.inter_op_num_threads = 1
    tok  = AutoTokenizer.from_pretrained(dst_dir)
    ort  = ORTModelForSeq2SeqLM.from_pretrained(dst_dir, session_options=sess)
    print(f'\n  [{mname}]')
    all_ok = True
    for frag in PARTIAL_INPUTS:
        inp1 = tok([cfg['prefix'] + frag], return_tensors='pt', truncation=True, max_length=128)
        out  = ort.generate(**inp1, num_beams=1, max_new_tokens=64, repetition_penalty=1.3)
        hyp  = tok.batch_decode(out, skip_special_tokens=True)[0]
        empty   = len(hyp.strip()) == 0
        garbage = len(hyp) > len(frag) * 6
        status  = 'EMPTY' if empty else 'GARBAGE' if garbage else 'OK'
        if empty or garbage: all_ok = False
        print(f'    [{status}]  IN:"{frag}"  ->  "{hyp}"')
    if all_ok:
        print('    All partial inputs handled gracefully.')
    del ort, tok


── Exact-match rate (hyp == ref, case-insensitive) ──
| Model            | Format       | Exact   | Rate   |
|------------------|--------------|---------|--------|
| prithivida       | pytorch_fp32 | 29/50   | 58.0%  |
| prithivida       | onnx_fp32    | 29/50   | 58.0%  |
| prithivida       | onnx_int8    | 27/50   | 54.0%  |
| coedit-small     | pytorch_fp32 | 30/50   | 60.0%  |
| coedit-small     | onnx_fp32    | 30/50   | 60.0%  |
| coedit-small     | onnx_int8    | 30/50   | 60.0%  |
| vennify-t5-base  | pytorch_fp32 | 29/50   | 58.0%  |
| vennify-t5-base  | onnx_fp32    | 29/50   | 58.0%  |
| vennify-t5-base  | onnx_int8    | 28/50   | 56.0%  |
| aventiq-t5-small | pytorch_fp32 | 8/50    | 16.0%  |
| aventiq-t5-small | onnx_fp32    | 8/50    | 16.0%  |
| aventiq-t5-small | onnx_int8    | 11/50   | 22.0%  |
| visheratin-mini  | pytorch_fp32 | 23/50   | 46.0%  |
| visheratin-mini  | onnx_fp32    | 23/50   | 46.0%  |
| visheratin-mini  | onnx_int8    | 23/50   | 46.0%  |
| visherati

## Step 9 — Latency distribution plots (CDF + bar)

In [ ]:
# ── Step 9 — Latency CDF + bar charts ────────────────────────────────────────
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np

FORMATS       = ['pytorch_fp32', 'onnx_fp32', 'onnx_int8']
FORMAT_COLORS = {'pytorch_fp32': '#4C72B0', 'onnx_fp32': '#DD8452', 'onnx_int8': '#55A868'}
FORMAT_LABELS = {'pytorch_fp32': 'PyTorch FP32', 'onnx_fp32': 'ONNX FP32', 'onnx_int8': 'ONNX INT8'}
MODEL_LIST    = list(MODEL_REGISTRY.keys())
n_models      = len(MODEL_LIST)

fig, axes = plt.subplots(2, n_models, figsize=(5 * n_models, 9))
if n_models == 1:
    axes = axes.reshape(2, 1)
fig.suptitle('Latency Distribution — All Models x Formats', fontsize=14, fontweight='bold')

for col, mname in enumerate(MODEL_LIST):
    ax_cdf = axes[0][col]
    ax_bar = axes[1][col]

    for fmt in FORMATS:
        r = RESULTS[mname].get(fmt)
        if r is None: continue
        lats = sorted(r['lats'])
        y    = np.linspace(0, 1, len(lats))
        ax_cdf.plot(lats, y, color=FORMAT_COLORS[fmt],
                    label=FORMAT_LABELS[fmt], linewidth=2)
    ax_cdf.axvline(500, color='red', linestyle='--', alpha=0.6, label='500ms target')
    ax_cdf.set_title(mname, fontsize=10)
    ax_cdf.set_xlabel('Latency (ms)')
    if col == 0: ax_cdf.set_ylabel('CDF')
    ax_cdf.legend(fontsize=7); ax_cdf.set_xlim(0); ax_cdf.grid(True, alpha=0.3)

    x = np.arange(3); width = 0.25
    for fi, fmt in enumerate(FORMATS):
        r = RESULTS[mname].get(fmt)
        if r is None: continue
        ax_bar.bar(x + fi*width, [r['p50'], r['p95'], r['p99']],
                   width, color=FORMAT_COLORS[fmt], label=FORMAT_LABELS[fmt], alpha=0.85)
    ax_bar.axhline(500, color='red', linestyle='--', alpha=0.6)
    ax_bar.set_xticks(x + width); ax_bar.set_xticklabels(['P50','P95','P99'])
    if col == 0: ax_bar.set_ylabel('ms')
    ax_bar.legend(fontsize=7); ax_bar.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('/content/latency_comparison.png', dpi=120, bbox_inches='tight')
plt.show()
print('Chart saved: /content/latency_comparison.png')

Chart saved: /content/latency_comparison.png


## Step 10 — Quality degradation heatmap (PT-FP32 → ONNX-INT8)

In [ ]:
# ── Step 10 — Per-sentence chrF++ delta heatmap ───────────────────────────────
# Red = regression, Green = improvement vs PyTorch FP32 baseline.
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import sacrebleu, numpy as np

all_ref   = [p[1] for p in TEST_PAIRS]
MODEL_LIST = list(MODEL_REGISTRY.keys())
n_models   = len(MODEL_LIST)

def sent_chrf(hyps, refs):
    return [sacrebleu.sentence_chrf(h, [r]).score for h, r in zip(hyps, refs)]

fig, axes = plt.subplots(1, n_models, figsize=(5 * n_models, 8), sharey=True)
if n_models == 1:
    axes = [axes]
fig.suptitle(
    'Per-sentence chrF++ Delta: ONNX INT8 vs PyTorch FP32\n'
    '(red=regression, green=improvement, white=no change)',
    fontsize=12, fontweight='bold')

for col, mname in enumerate(MODEL_LIST):
    pt_hyps = RESULTS[mname]['pytorch_fp32']['hyps']
    i8      = RESULTS[mname].get('onnx_int8')
    if i8 is None: continue
    i8_hyps  = i8['hyps']
    pt_scores = sent_chrf(pt_hyps, all_ref)
    i8_scores = sent_chrf(i8_hyps, all_ref)
    deltas    = [i - p for i, p in zip(i8_scores, pt_scores)]
    grid      = np.array(deltas).reshape(10, 5)
    ax        = axes[col]
    im        = ax.imshow(grid, cmap='RdYlGn', vmin=-20, vmax=20, aspect='auto')
    ax.set_title(mname, fontsize=10)
    ax.set_xlabel('Sentence col (0-4)')
    if col == 0: ax.set_ylabel('Sentence row (0-9)')
    for i in range(10):
        for j in range(5):
            v = grid[i, j]
            color = 'black' if abs(v) < 15 else 'white'
            ax.text(j, i, f'{v:.0f}', ha='center', va='center', fontsize=7, color=color)

plt.colorbar(im, ax=axes[-1], label='chrF++ delta (INT8 - FP32)')
plt.tight_layout()
plt.savefig('/content/quality_heatmap.png', dpi=120, bbox_inches='tight')
plt.show()
print('Heatmap saved: /content/quality_heatmap.png')

Heatmap saved: /content/quality_heatmap.png


## Step 11 — Paired bootstrap significance test (chrF++, 1000 resamples)

In [ ]:
# ── Step 11 — Bootstrap significance test ─────────────────────────────────────
# Tests whether quality differences between variants are statistically real.
# p < 0.05: significant.  p >= 0.05: likely noise.
import random, sacrebleu
from tabulate import tabulate

all_ref = [p[1] for p in TEST_PAIRS]

def bootstrap_chrf(hyps1, hyps2, refs, n_samples=1000):
    rng = random.Random(42)
    n   = len(refs)
    diffs = []
    for _ in range(n_samples):
        idxs   = [rng.randint(0, n-1) for _ in range(n)]
        s_refs = [[refs[i] for i in idxs]]
        s_h1   = [hyps1[i] for i in idxs]
        s_h2   = [hyps2[i] for i in idxs]
        diffs.append(
            sacrebleu.corpus_chrf(s_h1, s_refs).score -
            sacrebleu.corpus_chrf(s_h2, s_refs).score
        )
    diffs.sort()
    lower = sum(d < 0 for d in diffs) / n_samples
    upper = sum(d > 0 for d in diffs) / n_samples
    return 2 * min(lower, upper)

FORMATS    = ['pytorch_fp32', 'onnx_fp32', 'onnx_int8']
MODEL_LIST = list(MODEL_REGISTRY.keys())

print('Running bootstrap tests (1000 resamples each) ...')
rows = []

# Within-model: format-to-format
pairs = [('pytorch_fp32','onnx_fp32'), ('pytorch_fp32','onnx_int8'), ('onnx_fp32','onnx_int8')]
for mname in MODEL_LIST:
    for f1, f2 in pairs:
        r1 = RESULTS[mname].get(f1)
        r2 = RESULTS[mname].get(f2)
        if r1 is None or r2 is None: continue
        p      = bootstrap_chrf(r1['hyps'], r2['hyps'], all_ref)
        winner = f1 if r1['chrf'] >= r2['chrf'] else f2
        sig    = 'SIG' if p < 0.05 else 'ns'
        rows.append([mname, f1, f2,
                     f"{r1['chrf']:.2f}", f"{r2['chrf']:.2f}",
                     f"{r1['chrf']-r2['chrf']:+.2f}",
                     f'{p:.4f}', sig, winner])

# Cross-model: best format per model (onnx_int8)
for i in range(len(MODEL_LIST)):
    for j in range(i+1, len(MODEL_LIST)):
        m1, m2 = MODEL_LIST[i], MODEL_LIST[j]
        r1 = RESULTS[m1].get('onnx_int8')
        r2 = RESULTS[m2].get('onnx_int8')
        if r1 is None or r2 is None: continue
        p      = bootstrap_chrf(r1['hyps'], r2['hyps'], all_ref)
        winner = m1 if r1['chrf'] >= r2['chrf'] else m2
        sig    = 'SIG' if p < 0.05 else 'ns'
        rows.append(['cross-model', f'{m1}/int8', f'{m2}/int8',
                     f"{r1['chrf']:.2f}", f"{r2['chrf']:.2f}",
                     f"{r1['chrf']-r2['chrf']:+.2f}",
                     f'{p:.4f}', sig, winner])

print(tabulate(rows,
    headers=['Scope','A','B','chrF(A)','chrF(B)','Delta','p-value','Sig','Winner'],
    tablefmt='github'))
print()
print('p < 0.05 -> statistically significant (95% confidence)')
print('p >= 0.05 -> difference may be sampling noise')


Running bootstrap tests (1000 resamples each) ...
| Scope            | A                     | B                     |   chrF(A) |   chrF(B) |   Delta |   p-value | Sig   | Winner           |
|------------------|-----------------------|-----------------------|-----------|-----------|---------|-----------|-------|------------------|
| prithivida       | pytorch_fp32          | onnx_fp32             |     86.4  |     86.4  |    0    |     0     | SIG   | pytorch_fp32     |
| prithivida       | pytorch_fp32          | onnx_int8             |     86.4  |     84.79 |    1.61 |     0.206 | ns    | pytorch_fp32     |
| prithivida       | onnx_fp32             | onnx_int8             |     86.4  |     84.79 |    1.61 |     0.206 | ns    | onnx_fp32        |
| coedit-small     | pytorch_fp32          | onnx_fp32             |     86.84 |     86.84 |    0    |     0     | SIG   | pytorch_fp32     |
| coedit-small     | pytorch_fp32          | onnx_int8             |     86.84 |     86.69 |    0.

## Step 12 — Final go/no-go scorecard

In [ ]:
# ── Step 12 — Deployment scorecard ───────────────────────────────────────────
from tabulate import tabulate

GATES = {
    'P50 < 500ms':    lambda r, ref: r['p50'] < 500,
    'P95 < 1500ms':   lambda r, ref: r['p95'] < 1500,
    'DELTA_BLEU > -2':    lambda r, ref: (r['bleu'] - ref['bleu']) > -2.0,
    'DELTA_chrF > -1.5':  lambda r, ref: (r['chrf'] - ref['chrf']) > -1.5,
    'Disk < 500MB':   lambda r, ref: r.get('disk_mb', 9999) < 500,
    'No truncation':  lambda r, ref: all(
        len(h) >= len(s) * 0.3
        for s, h in zip([p[0] for p in TEST_PAIRS], r['hyps'])),
}

gate_names = list(GATES.keys())
rows = []

for mname in MODEL_REGISTRY:
    ref = RESULTS[mname]['pytorch_fp32']
    for fmt in ['onnx_fp32', 'onnx_int8']:
        r = RESULTS[mname].get(fmt)
        if r is None: continue
        gate_results = ['PASS' if fn(r, ref) else 'FAIL' for fn in GATES.values()]
        passed  = gate_results.count('PASS')
        verdict = 'GO' if passed == len(GATES) else (
                  'CONDITIONAL' if passed >= len(GATES)-1 else 'NO-GO')
        rows.append([mname, fmt] + gate_results + [f'{passed}/{len(GATES)}', verdict])

print('=' * 100)
print('DEPLOYMENT SCORECARD — 3 Models x 2 ONNX Formats')
print('=' * 100)
print(tabulate(rows, headers=['Model','Format'] + gate_names + ['Score','Verdict'],
               tablefmt='github'))
print()
print('GO         -> all gates passed, ready for production')
print('CONDITIONAL -> one gate failed, review before shipping')
print('NO-GO      -> multiple gates failed, do not ship')


DEPLOYMENT SCORECARD — 3 Models x 2 ONNX Formats
| Model            | Format    | P50 < 500ms   | P95 < 1500ms   | DELTA_BLEU > -2   | DELTA_chrF > -1.5   | Disk < 500MB   | No truncation   | Score   | Verdict     |
|------------------|-----------|---------------|----------------|-------------------|---------------------|----------------|-----------------|---------|-------------|
| prithivida       | onnx_fp32 | FAIL          | PASS           | PASS              | PASS                | FAIL           | PASS            | 4/6     | NO-GO       |
| prithivida       | onnx_int8 | PASS          | PASS           | FAIL              | FAIL                | PASS           | PASS            | 4/6     | NO-GO       |
| coedit-small     | onnx_fp32 | PASS          | PASS           | PASS              | PASS                | FAIL           | PASS            | 5/6     | CONDITIONAL |
| coedit-small     | onnx_int8 | PASS          | PASS           | PASS              | PASS                | PASS    

## Step 13 — Package all INT8 models for download

In [ ]:
# ── Step 13 — Package all INT8 models ────────────────────────────────────────
import shutil, os

for mname in MODEL_REGISTRY:
    src_dir  = f'{EXPORT_ROOT}/{mname}/onnx_int8'
    zip_path = f'/content/{mname}_int8'
    shutil.make_archive(zip_path, 'zip', src_dir)
    mb = os.path.getsize(zip_path + '.zip') / 1e6
    print(f'  {mname}_int8.zip  ({mb:.0f} MB)')

print()
print('Drop each zip into your app as:')
print('  models/<model_name>/')
print('    encoder_model.onnx')
print('    decoder_model.onnx')
print('    decoder_with_past_model.onnx')
print('    tokenizer files')


  prithivida_int8.zip  (267 MB)
  coedit-small_int8.zip  (90 MB)
  vennify-t5-base_int8.zip  (266 MB)
  aventiq-t5-small_int8.zip  (90 MB)
  visheratin-mini_int8.zip  (58 MB)
  visheratin-tiny_int8.zip  (40 MB)
  pszemraj-small_int8.zip  (109 MB)

Drop each zip into your app as:
  models/<model_name>/
    encoder_model.onnx
    decoder_model.onnx
    decoder_with_past_model.onnx
    tokenizer files


In [ ]:
from huggingface_hub import list_models

keywords = ["grammar", "gec", "grammar-correction", "grammatical-error"]
seen = set()

for kw in keywords:
    for m in list_models(
        search=kw,
        sort="downloads",
        limit=20
    ):
        if m.id not in seen:
            seen.add(m.id)
            tags = ", ".join(m.tags[:5]) if m.tags else ""
            print(f"{m.downloads:>8,} dl  |  {m.id}  |  {tags}")

print(f"\nTotal unique models found: {len(seen)}")

 174,991 dl  |  grammarly/coedit-large  |  transformers, pytorch, safetensors, t5, text2text-generation
 120,705 dl  |  vennify/t5-base-grammar-correction  |  transformers, pytorch, t5, text2text-generation, grammar
  99,752 dl  |  prithivida/grammar_error_correcter_v1  |  transformers, pytorch, t5, text2text-generation, text-generation-inference
   5,424 dl  |  abdulmatinomotoso/English_Grammar_Checker  |  transformers, pytorch, tensorboard, bert, text-classification
   1,841 dl  |  pszemraj/grammar-synthesis-small  |  transformers, pytorch, onnx, safetensors, t5
   1,727 dl  |  pszemraj/flan-t5-large-grammar-synthesis  |  transformers, pytorch, onnx, safetensors, gguf
     723 dl  |  MRNH/mbart-italian-grammar-corrector  |  transformers, pytorch, mbart, text2text-generation, grammatical error correction
     711 dl  |  MRNH/mbart-german-grammar-corrector  |  transformers, pytorch, mbart, text2text-generation, grammatical error correction
     552 dl  |  visheratin/t5-efficient-mini-g

In [ ]:
!pip install -q huggingface_hub
!unzip -o /content/hf_upload_kit.zip -d /content/

Archive:  /content/hf_upload_kit.zip
   creating: /content/readmes/
  inflating: /content/readmes/aventiq-t5-small_README.md  
  inflating: /content/readmes/visheratin-mini_README.md  
  inflating: /content/readmes/pszemraj-small_README.md  
  inflating: /content/readmes/visheratin-tiny_README.md  
  inflating: /content/readmes/coedit-small_README.md  
  inflating: /content/readmes/prithivida_README.md  
  inflating: /content/readmes/vennify-t5-base_README.md  
  inflating: /content/upload_all.sh  


In [ ]:
from huggingface_hub import HfApi, create_repo, notebook_login
import shutil, os

notebook_login()

In [ ]:
from huggingface_hub import HfApi, create_repo
import shutil, os

# Paste your WRITE token here directly
WRITE_TOKEN = "hf_REDACTED"  # <-- paste your write token

api = HfApi(token=WRITE_TOKEN)

# Verify it's write
info = api.whoami()
print("Role:", info['auth']['accessToken']['role'])

Role: write


In [ ]:
from huggingface_hub import HfApi, create_repo
import shutil, os
YOUR_USERNAME = "TonyRaju"  # <-- CHANGE THIS

models = [
    ("prithivida",       "prithivida-grammar-correcter-onnx-int8",              "prithivida_README.md"),
    ("coedit-small",     "gec-t5-small-coedit-onnx-int8",                      "coedit-small_README.md"),
    ("vennify-t5-base",  "vennify-t5-base-grammar-correction-onnx-int8",       "vennify-t5-base_README.md"),
    ("aventiq-t5-small", "aventiq-t5-small-grammar-correction-onnx-int8",      "aventiq-t5-small_README.md"),
    ("visheratin-mini",  "visheratin-t5-mini-grammar-correction-onnx-int8",    "visheratin-mini_README.md"),
    ("visheratin-tiny",  "visheratin-t5-tiny-grammar-correction-onnx-int8",    "visheratin-tiny_README.md"),
    ("pszemraj-small",   "pszemraj-grammar-synthesis-small-onnx-int8",         "pszemraj-small_README.md"),
]

EXPORT_ROOT = "/content/gec_exports"
api = HfApi()

for model_key, repo_name, readme_file in models:
    repo_id = f"{YOUR_USERNAME}/{repo_name}"
    int8_dir = f"{EXPORT_ROOT}/{model_key}/onnx_int8"

    print(f"\n  Uploading {model_key}...")

    create_repo(repo_id, repo_type="model", exist_ok=True)

    readme_src = f"/content/readmes/{readme_file}"
    if os.path.exists(readme_src):
        shutil.copy2(readme_src, f"{int8_dir}/README.md")
    else:
        print(f"  ⚠️  README not found: {readme_src}")

    api.upload_folder(
        folder_path=int8_dir,
        repo_id=repo_id,
        commit_message="Add ONNX INT8 quantized model with benchmarks",
    )
    print(f"  ✅ https://huggingface.co/{repo_id}")

print("\n  All 7 models uploaded!")


  Uploading prithivida...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/hf_api.py:3664: UserWarning: Warnings while validating metadata in README.md:
- The pipeline tag "text2text-generation" is not in the official list: text-classification, token-classification, table-question-answering, question-answering, zero-shot-classification, translation, summarization, feature-extraction, text-generation, fill-mask, sentence-similarity, text-to-speech, text-to-audio, automatic-speech-recognition, audio-to-audio, audio-classification, audio-text-to-text, voice-activity-detection, depth-estimation, image-classification, object-detection, image-segmentation, text-to-image, image-to-text, image-to-image, image-to-video, unconditional-image-generation, video-classification, reinforcement-learning, robotics, tabular-classification, tabular-regression, tabular-to-text, table-to-text, multiple-choice, text-ranking, text-retrieval, time-series-forecasting, text-to-video, image-text-to-text, image-text-to-image, image-

decoder_model.onnx:   0%|          | 0.00/163M [00:00<?, ?B/s]

encoder_model.onnx:   0%|          | 0.00/110M [00:00<?, ?B/s]

decoder_with_past_model.onnx:   0%|          | 0.00/149M [00:00<?, ?B/s]

...

decoder_with_past_model.onnx:   0%|          | 0.00/55.4M [00:00<?, ?B/s]

encoder_model.onnx:   0%|          | 0.00/35.6M [00:00<?, ?B/s]

Upload 4 LFS files:   0%|          | 0/4 [00:00<?, ?it/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

  ✅ https://huggingface.co/TonyRaju/pszemraj-grammar-synthesis-small-onnx-int8

  All 7 models uploaded!


In [ ]:
from huggingface_hub import whoami
info = whoami()
print(f"Username: {info['name']}")
print(f"Type: {info['type']}")

In [ ]:
from huggingface_hub import HfApi
api = HfApi()
info = api.whoami()
print(info)

In [ ]:
from huggingface_hub import HfApi

api = HfApi()
try:
    info = api.whoami()
    print("Full response:", info)
    print("Name:", info.get("name"))
except Exception as e:
    print("Error:", e)

Full response: {'type': 'user', 'id': '6891c40a5b6d47986c2d865a', 'name': 'TonyRaju', 'fullname': 'Anthony Raju Kondaveeti', 'email': 'tonykondaveetijmj98@gmail.com', 'emailVerified': True, 'canPay': False, 'billingMode': 'prepaid', 'periodEnd': 1777593600, 'isPro': False, 'avatarUrl': '/avatars/0fa7cfaffda6a14aaac8ad455f2f5f2d.svg', 'orgs': [{'type': 'org', 'id': '689ececa6a3274a441c01ebd', 'name': 'Gen-AI-Project', 'fullname': 'Gen-AI', 'email': None, 'canPay': False, 'billingMode': 'postpaid', 'periodEnd': None, 'avatarUrl': 'https://www.gravatar.com/avatar/56f511110e0920badfaa88b0a1094261?d=retro&size=100', 'roleInOrg': 'contributor'}], 'auth': {'type': 'access_token', 'accessToken': {'displayName': 'SmartKey', 'role': 'read', 'createdAt': '2026-04-02T12:23:19.846Z'}}}
Name: TonyRaju


In [ ]:
!huggingface-cli whoami

In [ ]:
for model_key, repo_name, readme_file in models:
    repo_id = f"{YOUR_USERNAME}/{repo_name}"
    int8_dir = f"{EXPORT_ROOT}/{model_key}/onnx_int8"
    readme_path = f"{int8_dir}/README.md"

    with open(readme_path, "r") as f:
        content = f.read()
    content = content.replace("pipeline_tag: text2text-generation", "pipeline_tag: text2text")
    with open(readme_path, "w") as f:
        f.write(content)

    api.upload_file(
        path_or_fileobj=readme_path,
        path_in_repo="README.md",
        repo_id=repo_id,
        commit_message="Fix pipeline_tag metadata",
    )
    print(f"  ✅ Fixed {repo_id}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/hf_api.py:3664: UserWarning: Warnings while validating metadata in README.md:
- The pipeline tag "text2text" is not in the official list: text-classification, token-classification, table-question-answering, question-answering, zero-shot-classification, translation, summarization, feature-extraction, text-generation, fill-mask, sentence-similarity, text-to-speech, text-to-audio, automatic-speech-recognition, audio-to-audio, audio-classification, audio-text-to-text, voice-activity-detection, depth-estimation, image-classification, object-detection, image-segmentation, text-to-image, image-to-text, image-to-image, image-to-video, unconditional-image-generation, video-classification, reinforcement-learning, robotics, tabular-classification, tabular-regression, tabular-to-text, table-to-text, multiple-choice, text-ranking, text-retrieval, time-series-forecasting, text-to-video, image-text-to-text, image-text-to-image, image-text-to-vid

  ✅ Fixed TonyRaju/prithivida-grammar-correcter-onnx-int8
  ✅ Fixed TonyRaju/gec-t5-small-coedit-onnx-int8
  ✅ Fixed TonyRaju/vennify-t5-base-grammar-correction-onnx-int8
  ✅ Fixed TonyRaju/aventiq-t5-small-grammar-correction-onnx-int8
  ✅ Fixed TonyRaju/visheratin-t5-mini-grammar-correction-onnx-int8
  ✅ Fixed TonyRaju/visheratin-t5-tiny-grammar-correction-onnx-int8
  ✅ Fixed TonyRaju/pszemraj-grammar-synthesis-small-onnx-int8


In [ ]:
for model_key, repo_name, readme_file in models:
    repo_id = f"{YOUR_USERNAME}/{repo_name}"
    int8_dir = f"{EXPORT_ROOT}/{model_key}/onnx_int8"
    readme_path = f"{int8_dir}/README.md"

    with open(readme_path, "r") as f:
        content = f.read()
    content = content.replace("pipeline_tag: text2text\n", "")
    content = content.replace("pipeline_tag: text2text-generation\n", "")
    with open(readme_path, "w") as f:
        f.write(content)

    api.upload_file(
        path_or_fileobj=readme_path,
        path_in_repo="README.md",
        repo_id=repo_id,
        commit_message="Remove invalid pipeline_tag",
    )
    print(f"  ✅ Fixed {repo_id}")

  ✅ Fixed TonyRaju/prithivida-grammar-correcter-onnx-int8
  ✅ Fixed TonyRaju/gec-t5-small-coedit-onnx-int8
  ✅ Fixed TonyRaju/vennify-t5-base-grammar-correction-onnx-int8
  ✅ Fixed TonyRaju/aventiq-t5-small-grammar-correction-onnx-int8
  ✅ Fixed TonyRaju/visheratin-t5-mini-grammar-correction-onnx-int8
  ✅ Fixed TonyRaju/visheratin-t5-tiny-grammar-correction-onnx-int8
  ✅ Fixed TonyRaju/pszemraj-grammar-synthesis-small-onnx-int8
